In [1]:
import pandas as pd
from rapidfuzz import fuzz
import spacy

nlp = spacy.load("en_core_web_md")
statistic_types = {'CARDINAL', 'PERCENT', 'MONEY', 'QUANTITY'}

def stat_counter(text):
    if not isinstance(text, str):
        return 0
    doc = nlp(text)
    return sum(ent.label_ in statistic_types for ent in doc.ents)

conservative_bigrams = pd.read_csv('../data/top_conservative_bigrams.csv')['bigram'].tolist()
liberal_bigrams = pd.read_csv('../data/top_liberal_bigrams.csv')['bigram'].tolist()

def match_counter(statement, bigram_list, threshold=70):
    words = [word.text.lower() for word in nlp(str(statement))]
    bigram_coll = [''.join(words[i:i+2]) for i in range(len(words)-1)]
    matches = 0
    for bigram in bigram_coll:
        for check in bigram_list:
            if fuzz.ratio(bigram, check) >= threshold:
                matches += 1
                break
    return matches

def political_bias_counts(statement):
    stat_count = stat_counter(statement)
    cons_bigram_count = match_counter(statement, conservative_bigrams)
    lib_bigram_count = match_counter(statement, liberal_bigrams)

    return {
        "stat_count": stat_count,
        "conservative_bigram_count": cons_bigram_count,
        "liberal_bigram_count": lib_bigram_count
    }


In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def emotional_intensity_score(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    vs = analyzer.polarity_scores(text)
    return abs(vs['compound'])


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

spam_model_name = "mrm8488/bert-tiny-finetuned-sms-spam-detection"
spam_tokenizer = AutoTokenizer.from_pretrained(spam_model_name)
spam_model = AutoModelForSequenceClassification.from_pretrained(spam_model_name)
spam_model.eval()

def spam_probability(text):
    inputs = spam_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )
    with torch.no_grad():
        outputs = spam_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        return probs[0, 1].item()  # probability that it's spam




c:\Users\Chris Mo\Documents\GitHub\ai4good\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd
import numpy as np 
import sys 
import torch
sys.path.append('../utils/')
from mxnet_utils import *
from nlp_utils import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


In [5]:
import pickle

with open("../data/vocabs.pkl", "rb") as f:
    vocabs = pickle.load(f)

topic_vocab       = vocabs["topic_vocab"]
author_vocab      = vocabs["author_vocab"]
job_vocab         = vocabs["job_vocab"]
location_vocab    = vocabs["location_vocab"]
affiliation_vocab = vocabs["affiliation_vocab"]
label_map         = vocabs["label_map"]

print("Loaded vocabs and label_map.")

Loaded vocabs and label_map.


In [13]:
import torch
import torch.nn.functional as F
from transformers import BertTokenizer


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

pretrained_state_dict = torch.load("../checkpoints/best.pth", map_location="cpu")

def run_truth_model(
    net,
    tokenizer,
    article_text,
    topic_vec=None,
    author_id=None,
    job_id=None,
    loc_id=None,
    aff_id=None,
    history_vec=None,
    device="cpu"
):
    if topic_vec is None: topic_vec = torch.rand(1, 60)
    if author_id is None: author_id = torch.tensor([1])
    if job_id is None: job_id = torch.tensor([1])
    if loc_id is None: loc_id = torch.tensor([1])
    if aff_id is None: aff_id = torch.tensor([1])
    if history_vec is None: history_vec = torch.rand(1, 6)

    net.eval()

    # ---------------------------
    # 1. Tokenize text
    # ---------------------------
    encoding = tokenizer(
        article_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512
    )

    input_ids = encoding["input_ids"].to(device)
    token_types = encoding.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # ---------------------------
    # 2. Convert metadata to tensors
    # ---------------------------

    # Topic vector: should be shape (1, num_topics)
    if isinstance(topic_vec, np.ndarray):
        topic_vec = torch.tensor(topic_vec, dtype=torch.float32).unsqueeze(0)
    if topic_vec.dim() == 1:
        topic_vec = topic_vec.unsqueeze(0)
    topic_vec = topic_vec.to(device)

    # Categorical IDs
    author_id = torch.tensor([author_id]).to(device)
    job_id = torch.tensor([job_id]).to(device)
    loc_id = torch.tensor([loc_id]).to(device)
    aff_id = torch.tensor([aff_id]).to(device)

    # History vector (6-dim)
    if isinstance(history_vec, np.ndarray):
        history_vec = torch.tensor(history_vec, dtype=torch.float32).unsqueeze(0)
    if history_vec.dim() == 1:
        history_vec = history_vec.unsqueeze(0)
    history_vec = history_vec.to(device)

    # ---------------------------
    # 3. Forward pass
    # ---------------------------
    with torch.no_grad():
        logits = net(
            input_ids,
            token_types,
            attention_mask,
            topic_vec,
            author_id,
            job_id,
            loc_id,
            aff_id,
            history_vec
        )

        probs = F.softmax(logits, dim=-1)
        pred_idx = probs.argmax(dim=-1).item()

    # ---------------------------
    # 4. Convert to label string
    # ---------------------------
    class_labels = [
        "false",
        "half-true",
        "mostly-true",
        "true",
        "barely-true",
        "pants-fire"
    ]
    pred_label = class_labels[pred_idx]

    return {
        "logits": logits,
        "probabilities": probs,
        "predicted_label": pred_label,
        "predicted_index": pred_idx
    }


In [14]:
import random
from transformers import BertModel, BertTokenizer, BertConfig

np.random.seed(100)
torch.manual_seed(100)
random.seed(100)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(100)

bert_dropout = 0.1 # From later in the notebook
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
config = BertConfig.from_pretrained('bert-base-uncased',
                                    hidden_dropout_prob=bert_dropout,
                                    attention_probs_dropout_prob=bert_dropout)
bert_base = BertModel.from_pretrained('bert-base-uncased', config=config)
bert_base.to(device)
print(bert_base)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [15]:
label_map = {
    'false': 0,
    'half-true': 1,
    'mostly-true': 2,
    'true': 3,
    'barely-true': 4,
    'pants-fire': 5
}

net_best2 = BERTClassifier(
    bert=bert_base,
    num_topics=len(topic_vocab),
    num_authors=len(author_vocab),
    num_jobs=len(job_vocab),
    num_locations=len(location_vocab),
    num_affiliations=len(affiliation_vocab),
    num_classes=len(label_map),
    embed_dim=32,
    author_dropout=0.1,
    author_mlp_layers=2,
    author_mlp_hidden=192,
    history_dropout=0.1,
    history_mlp_layers=2,
    history_mlp_hidden=192)
net_best2.load_state_dict(torch.load('../checkpoints/best.pth', map_location=device))
net_best2.to(device)


BERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [16]:
def analyze_for_genai(article_text):
    truth_result = run_truth_model(net_best2, tokenizer, article_text)
    prob_tensor = truth_result["probabilities"].flatten().tolist()

    truth_probs = {
        "prob_false": prob_tensor[0],
        "prob_half_true": prob_tensor[1],
        "prob_mostly_true": prob_tensor[2],
        "prob_true": prob_tensor[3],
        "prob_barely_true": prob_tensor[4],
        "prob_pants_on_fire": prob_tensor[5]
        # "prob_nan": prob_tensor[6]
    }

    bias_counts = political_bias_counts(article_text)
    bias_vector = [
        bias_counts["stat_count"],
        bias_counts["conservative_bigram_count"],
        bias_counts["liberal_bigram_count"]
    ]

    emotional_score = emotional_intensity_score(article_text)
    spam_score = spam_probability(article_text)

    feature_vector = list(truth_probs.values()) + bias_vector + [emotional_score, spam_score]

    return feature_vector

In [ ]:
import os
from openai import OpenAI
import json
import gradio as gr
import re
import yaml

from google import genai
from google.genai import types

with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config["gemini"]["api"]

def run_genai_with_vector(article_text, feature_vector):
    vector_str = ", ".join([f"{v:.3f}" if isinstance(v, float) else str(v) for v in feature_vector])

    prompt = f"""
You are an AI assistant assigned to evaluate the factuality of news statements using a generative fact-checking pipeline.
Your task is to analyze article text, incorporate predictive model outputs WITHOUT overweighting them, and compute factor scores using the scoring recipes below.

VECTOR DESCRIPTION
The feature vector contains the following predictive model outputs and auxiliary measures:
0-5: Probabilities for truthfulness classes from our custom BERT-based model:
     0 = False, 1 = Half True, 2 = Mostly True, 3 = True, 4 = Barely True, 5 = Pants on Fire
7: Count of numeric/statistical entities detected in the text
8: Count of conservative bigram matches in the text
9: Count of liberal bigram matches in the text
10: Emotional intensity score (absolute VADER compound score)
11: Spam likelihood score (0–1, probability of being spam)

ANTI-BIAS CONSTRAINT
- Treat predictive model scores only as auxiliary context.
- Do NOT give these predictive scores undue weight.
- If your analysis of the article text contradicts the predictive scores, rely on the TEXTUAL EVIDENCE and explain the discrepancy.

==========================
FACTUALITY FACTORS (6 TOTAL)
==========================

1. AUTHENTICITY
- Definition: Does the text present evidence that the claims are genuine, verifiable, and traceable?
- Scoring Recipe (1–10): Look for verifiable details, named sources, timestamps, data, official statements; higher when concrete and falsifiable.
- Output: numeric score + 1–2 sentences referencing evidence.

2. SENSATIONALISM
- Definition: Presence of hyperbole, emotional language, exaggeration.
- Scoring Recipe (1–10): Extract emotional/hyperbolic language, count dramatic constructions, score based on density and prominence.
- Output: score + 2 example phrases.

3. POLITICAL BIAS
- Definition: Degree to which the article leans left, center, or right.
- Scoring Recipe (0–10 + tag): Identify partisan framing or selective omission.
- Output: numeric score + category {{left, centrist, right, mixed}} + examples.

4. TOXICITY
- Definition: Hostile, demeaning, or aggressive language.
- Scoring Recipe (1–10): Identify insults, threats, aggression, and target.
- Output: score + most toxic example phrase.

5. CONFIRMATION BIAS
- Definition: Selective presentation of information reinforcing a preferred conclusion.
- Scoring Recipe (1–10): Identify cherry-picked evidence or missing counterarguments.
- Output: score + 1 example.

6. SHORT-TERM UTILITY (Profit Incentive)
- Definition: Degree content maximizes clicks or engagement.
- Scoring Recipe (1–10): Detect clickbait, urgent calls to action, monetization cues.
- Output: score + 1–2 indicators of profit-driven framing.

OUTPUT FORMAT (STRICT JSON)
{{
  "veracity_label": "One of: True, Mostly True, Half True, Mostly False, False, Pants on Fire",
  "explanation_text": "A well-detailed explanation explaining the final verdict and reconciling any discrepancies. Explain each factuality factor's score choice as well",
  "factor_scores": [
    {{"factor": "Authenticity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Sensationalism", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Political Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Toxicity", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Confirmation Bias", "score": 1-10, "reasoning": "Brief evidence"}},
    {{"factor": "Short-term Utility", "score": 1-10, "reasoning": "Brief evidence"}}
  ]
}}

ARTICLE TEXT:
\"\"\"
{article_text}
\"\"\"

PREDICTIVE MODEL FEATURE VECTOR:
[{vector_str}]
"""

    # client = OpenAI(
    #     api_key = api_key, 
    #     base_url = "https://ellm.nrp-nautilus.io/v1"
    # )

    # completion = client.chat.completions.create(
    #     model="gemma3",
    #     messages=[
          
    #         {"role": "system", "content": "You are a helpful assistant that outputs STRICT JSON only."},
    #         {"role": "user", "content": prompt},
    #     ],
    #     temperature=0.1 
    # )

    # return completion.choices[0].message.content
    client = genai.Client(
        api_key = api_key
    )

    completion = client.models.generate_content(
        model="gemini-2.5-pro",
        config=types.GenerateContentConfig(
            system_instruction="You are a helpful assistant that outputs STRICT JSON only."),
        contents=prompt
    )

    return completion.text

def run_full_pipeline(article_text, progress=gr.Progress(track_tqdm=False)):
    if not article_text.strip():
        return "Error", [], "Please enter text.", {}

    try:
        progress(0.1, "Analyzing Features...")
        feature_vector = analyze_for_genai(article_text)

        progress(0.5, "Consulting LLM...")
        
        json_response_str = run_genai_with_vector(article_text, feature_vector)
        
        print(f"DEBUG: Raw Model Output:\n{json_response_str}") # Check your console if errors happen

        clean_str = re.sub(r"```json|```", "", json_response_str).strip()
        
        if "{" in clean_str:
            start = clean_str.find("{")
            end = clean_str.rfind("}") + 1
            clean_str = clean_str[start:end]

        data = json.loads(clean_str)

        veracity = data.get("veracity_label", "Unknown")
        scores_list = data.get("factor_scores", [])
        
        if not isinstance(scores_list, list):
             scores_list = []

        df_data = [[item["factor"], item["score"], item["reasoning"]] for item in scores_list]
        explanation = data.get("explanation_text", "")

        vector_display = {}
        if len(feature_vector) >= 11:
            labels = [
                "False (0)", 
                "Half True (1)", 
                "Mostly True (2)", 
                "True (3)", 
                "Barely True (4)", 
                "Pants on Fire (5)" 
                # "NaN (6)"
            ]
            prob_map = {label: round(feature_vector[i], 4) for i, label in enumerate(labels)}
            vector_display = {
                "Truth Model Breakdown": prob_map,
                "Numeric Entities": feature_vector[5],
                "Conservative Bigrams": feature_vector[6],
                "Liberal Bigrams": feature_vector[7],
                "Emotional Intensity": feature_vector[8],
                "Spam Score": feature_vector[9]
            }

        return veracity, df_data, explanation, vector_display

    except Exception as e:
        error_msg = f"System Error: {str(e)}"
        print(error_msg)
        return "Error", [], error_msg, {}

with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="red")) as demo:
    gr.Markdown("# News Fact-Analysis")
    gr.Markdown("Paste an article below.")

    with gr.Row():
        with gr.Column(scale=1):
            article_input = gr.Textbox(
                label="Input Article", 
                placeholder="Paste text here...", 
                lines=15
            )
            submit_btn = gr.Button("Analyze Article", variant="primary")
        
        with gr.Column(scale=1):
            veracity_output = gr.Label(label="Veracity Verdict")
            explanation_output = gr.Markdown("### Analysis Summary\n*Run analysis to see details.*")
            scores_output = gr.Dataframe(
                headers=["Factor", "Score", "Reasoning"],
                datatype=["str", "number", "str"],
                label="Detailed Factor Scores",
                wrap=True
            )
            vector_output = gr.JSON(label="Underlying Model Vector")

    submit_btn.click(
        fn=run_full_pipeline,
        inputs=[article_input],
        outputs=[veracity_output, scores_output, explanation_output, vector_output]
    )

if __name__ == "__main__":
    demo.queue()
    demo.launch()



* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DEBUG: Raw Model Output:
```json
{
  "veracity_label": "False",
  "explanation_text": "The article is a piece of satire and is not intended to be a factual report. The central claim that an MRI confirmed Donald Trump has 'incurable advanced-stage patriotism' is a medical and logical absurdity. The quotes attributed to a press secretary and Trump are fabricated for comedic effect, mimicking his known speech patterns. The predictive model's highest probability was 'True' (55.3%), which is a clear error. This discrepancy likely arises because the model was fooled by the article's structure, which mimics a genuine news report, while failing to grasp the satirical and impossible nature of the content itself. The textual evidence of fabrication is overwhelming, and therefore the model's prediction is disregarded.\n\nFactor scores are assigned as follows: Authenticity is minimal (1/10) because the claims are entirely unverifiable and fantastical. Sensationalism is high (9/10) due to the hyper